# 🏆 Interview Questions — System Design in FastAPI — Interview Questions

*Deep-dive Q&A for staff/principal-level interviews.*

**How to use:** Say your answer aloud, then read the model answer. If you can't name the failure mode and the trade-off, study the section again.

---
## 🏆 Interview Questions — System Design in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# System Design in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A "charge card" endpoint sometimes double-charges customers. What's the root cause and the fix?


**Deep dive.** The network guarantees *at-least-once* delivery: clients, load
balancers, and proxies all retry on timeout, and a request can succeed on the
server while the *response* is lost — the client then retries a charge that
already happened. With nothing to deduplicate on, the retry charges again. The
fix is an **idempotency key**: the client sends a unique key per logical
operation; the server performs the charge the first time and stores the response
keyed by it, then *replays* that stored response on any retry with the same key,
without re-charging. This converts at-least-once delivery into exactly-once
*effect*. Critically, persist the result *before* acknowledging, so a
crash-then-retry is still safe.

---

### Q2. Why is doing slow work synchronously on the request path an architectural problem, and what are the options for moving it off?


**Deep dive.** Slow inline work (PDF render, email, third-party call) inflates p99
latency, holds the worker/connection for its full duration (reducing throughput),
and couples the client's success to the slow task's success. Options, in
increasing robustness: **FastAPI `BackgroundTasks`** (simple, in-process — runs
after the response, but is lost if the process dies and doesn't survive restarts);
a **task queue** (Celery/RQ/Arq — durable, retryable, scalable across workers); or
**publishing an event** to a broker for a separate consumer (fully decoupled,
event-driven). The right choice depends on durability needs: `BackgroundTasks` for
best-effort fire-and-forget, a queue/broker when the work *must* eventually happen.

---

### Q3. Apply CAP to this payment service. During a network partition, what do you choose?


**Deep dive.** Payments demand correctness, so you choose **CP** (consistency over
availability): during a partition, it's better to reject or hold a transaction
than to risk a double-spend or a divergent ledger that must later be reconciled by
hand. That means the write path depends on a strongly-consistent store and will
return errors rather than accept ambiguous writes when it can't guarantee
correctness. The complement is PACELC: even without a partition, you're trading
latency for consistency — synchronous strong consistency costs p99 latency on
every charge, which you accept for money-handling. Contrast with a product-view
counter, where you'd pick AP and tolerate staleness.

---

### Q4. Where should the idempotency store live, what's its lifecycle, and what are the failure modes?


**Deep dive.** It should be a fast, shared, durable store — typically Redis or a
DB table — *shared across all instances*, because a per-process dict doesn't
deduplicate across a fleet (a retry hitting a different instance would re-charge).
Entries carry a **TTL** matched to the retry window (hours, not forever) to bound
memory. Failure modes: (a) storing the response *after* acking, so a crash between
charge and store loses the dedupe record — persist before returning; (b) racing
concurrent retries with the same key — use an atomic set-if-absent (`SET NX`) or a
DB unique constraint so only one wins; (c) storing a *failed* attempt as success —
be deliberate about whether errors are cached.

---

### Q5. Do a back-of-the-envelope estimate for this service (5M charges/day) and identify the bottleneck.


**Deep dive.** 5M / 86,400 ≈ 58 charges/sec average; design for peak at 3–5×, so
~200–300 writes/sec. That's a modest write rate a single well-tuned primary
handles, so the bottleneck is unlikely to be raw DB write throughput — it's more
likely the **synchronous downstream work** (payment provider latency, receipt
generation) inflating latency and tying up workers, plus correctness under retry.
The math tells you the design priorities here are *idempotency and offloading slow
work*, not sharding. The point of the estimate is to reveal that the scaling
problem is latency/correctness, not volume — so you don't over-engineer storage.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Networks provide which delivery guarantee by default, forcing idempotency?**
- A) exactly-once
- B) at-most-once
- C) at-least-once
- D) ordered-once

**2. An idempotency key makes a retried charge safe by:**
- A) encrypting the request
- B) replaying the stored response without re-executing the side effect
- C) rejecting all retries
- D) charging a smaller amount

**3. You must persist the idempotency result:**
- A) after returning the response
- B) before acknowledging the response
- C) only on failure
- D) never — keep it in memory

**4. A per-process in-memory idempotency dict fails in production because:**
- A) it's too slow
- B) it doesn't deduplicate across multiple server instances
- C) it violates SOLID
- D) dicts can't store responses

**5. For a payment service during a network partition, you choose:**
- A) AP — availability over consistency
- B) CP — consistency over availability
- C) neither — CAP doesn't apply
- D) both simultaneously

### Answer Key
1. **C** — at-least-once delivery is why retries duplicate.
2. **B** — replay the stored result; run the side effect once.
3. **B** — persist before acking so crash-then-retry is safe.
4. **B** — a local dict can't dedupe across a fleet; use a shared store.
5. **B** — payments favor correctness (CP).

---

## Part 3 — Gotchas Checklist

- **Assume at-least-once delivery.** Any endpoint with a side effect (charge,
  ship, post) needs an idempotency key or naturally-idempotent operation.
- **Persist the idempotency record BEFORE responding**, or a crash between the
  effect and the store re-runs the effect on retry.
- **Use a shared, TTL'd idempotency store** (Redis/DB), not a per-process dict —
  otherwise retries to other instances duplicate.
- **Guard concurrent retries** with an atomic set-if-absent / unique constraint,
  or two in-flight retries both execute.
- **`BackgroundTasks` are best-effort** and in-process: they run after the
  response but are lost on crash/restart. Use a durable queue when the work must
  happen.
- **Don't do slow/CPU-bound work on the request path** — it inflates p99 and
  starves workers.
- **Do the capacity math first** — it tells you whether the problem is volume
  (shard) or latency/correctness (idempotency + offload), preventing
  over-engineering.
- **Decide error caching explicitly** — caching a transient failure as the
  permanent result under an idempotency key is a nasty, subtle bug.

---
## Part 4 — Advanced Architecture Deep-Dive Questions
*(Scalability · HA · Quorum · CAP · Connection Pooling · Leader Election · Consensus · Backpressure · Idempotency · DLQ · Saga · API Gateway · Service Discovery)*


### Q6. ShopFlow is hitting 50k RPS. The CTO says "just get a bigger DB server." What's wrong with that plan, and what's your counter-proposal?

**Deep dive.** Vertical scaling (bigger server) works up to a ceiling and has three problems: (1) hardware cost grows exponentially past mid-tier — doubling RAM costs 4× the price; (2) one server = one point of failure — no redundancy; (3) upgrades require downtime. At 50k RPS the real bottleneck is almost certainly read-heavy traffic hitting the DB, not raw write throughput. Counter-proposal: **first exhaust vertical** (free wins), then add **read replicas** (route SELECTs there), then put **Redis in front** of the hot read paths — this alone often drops DB load by 80%. If writes are the bottleneck, then **shard by user_id/region**. Critically, the app tier must be stateless first (sessions in Redis, not server RAM) to allow horizontal autoscaling. The shape of the traffic (read vs write ratio, hot keys, query patterns) tells you which lever to pull — never blindly follow "bigger server."

---


### Q7. Your service has three DB replicas. A network partition isolates one replica. What happens, and why does "always use an odd number of nodes" matter?

**Deep dive.** With N=3 nodes and quorum = ⌊N/2⌋+1 = 2, the majority partition (2 nodes) can still reach quorum and continue accepting writes. The isolated node (1 node) cannot — it refuses to write, preventing split-brain (two sides making conflicting decisions). This is Raft/Paxos in practice. The "odd number" rule: N=4 gives you quorum=3 and still only survives 1 failure — identical fault-tolerance to N=3, but costs an extra server. N=5 quorum=3 survives 2 failures. Always go 3→5→7, never 2→4. The PACELC extension matters too: even without a partition, you choose between Latency and Consistency. Synchronous replication (strong C) adds latency on every write; asynchronous (lower L) risks data loss on failover. Money writes → sync at least one replica; analytics writes → async acceptable.

---

### Q8. Explain CAP. Where do your datastores live on the triangle, and how does it affect your design decisions?

**Deep dive.** CAP: in the presence of a network **P**artition, you choose **C**onsistency (all nodes see the same data) OR **A**vailability (every request gets a response — possibly stale). You can't have both. No real system eliminates partitions, so the real choice is C vs A during a split. Practical mapping: **PostgreSQL (CP)** — primary stops taking writes if it can't confirm replicas; **Cassandra (AP)** — every node always accepts writes (eventual consistency); **Redis Sentinel (CP)** — majority quorum required to elect new primary. Design implications: use CP (Postgres) for money/orders/inventory where correctness trumps availability; use AP (Cassandra/DynamoDB) for user activity, product views, metrics where staleness is acceptable. PACELC refines this: even without partitions, Cassandra trades Latency for Consistency; Spanner trades Latency for Consistency via TrueTime. Name the tradeoff explicitly in design reviews.

---

### Q9. A service keeps crashing because it opens 1,000 simultaneous DB connections at peak load. How do you fix this and size the pool?

**Deep dive.** Each PostgreSQL connection consumes ~5–10MB of server RAM + a backend process. 1,000 connections = 5–10GB just for connection overhead, starving the actual query executors. Fix: **connection pooling** — a pool of N reusable connections shared across all threads/coroutines. Requests acquire a connection, use it, release it; at most N queries run simultaneously. Sizing: PostgreSQL's own recommendation is `pool_size ≈ num_CPU_cores × 2 + num_effective_spindles`. For a 4-core DB: ~10 connections per app instance; with 20 app servers: 200 total — check this is under `max_connections / 2` to leave headroom for admin queries. For async (asyncio/asyncpg), the pool can be smaller because connections aren't blocked waiting for I/O. PgBouncer as a proxy pool is the production standard — it multiplexes thousands of app connections onto tens of DB connections transparently.

---

### Q10. Your Kafka consumer crashes after processing a message but before committing the offset. What happens on restart, and why does your handler need to be idempotent?

**Deep dive.** Kafka guarantees at-least-once delivery: the consumer restarts, sees the uncommitted offset, and re-delivers the same message. If your handler has side effects (charge card, send email, write to DB), the side effect runs twice. Solution: make the handler **idempotent**. Three approaches: (a) natural idempotency — the operation is inherently safe to repeat (e.g., `UPDATE SET status='processed' WHERE status='pending'` only changes state once); (b) idempotency key — store a `(event_id, result)` in Redis with `SET NX`; on retry, return the stored result without re-executing; (c) transactional outbox + deduplication table — persist processed event_id in the same DB transaction as the business write. The **Dead Letter Queue** handles the opposite case: if a message can never be processed (malformed payload, deleted entity), after N retries it moves to the DLQ for human inspection rather than blocking the partition indefinitely.

---

### Q11. Design the leader election mechanism for a distributed cron scheduler. What prevents two nodes from both believing they're the leader?

**Deep dive.** The zombie leader problem: a leader pauses (GC, network hiccup), the cluster elects a new leader, the old leader resumes believing it's still in charge — now you have two writers. Prevention: **fencing tokens** — every election issues a monotonically increasing integer (stored in etcd/ZooKeeper). Each write to shared state includes the token; storage rejects writes with a stale (lower) token. The leader must heartbeat within `election_timeout`; if it misses, followers hold an election. Raft mechanics: (1) follower times out → becomes candidate → increments term → sends RequestVote to all peers; (2) if majority vote YES (on the same term) → wins election, becomes leader; (3) terms act as logical clocks — any node seeing a higher term immediately becomes a follower. Production implementation: use etcd's `LeaseGrant` (TTL-based leader lock) + `KeepAlive` heartbeat rather than rolling your own — Kubernetes scheduler does exactly this.

---

### Q12. A payment processing worker is falling behind — the queue depth is growing faster than it drains. What's backpressure, and what are your options?

**Deep dive.** Backpressure = applying resistance upstream when downstream can't keep up, so the fast producer doesn't overwhelm the slow consumer. Without it: queue grows unboundedly → OOM → crash. Options in increasing severity: (1) **Block** — producer blocks when the queue hits capacity; simple, but propagates delay upstream (callers wait); good when delay is acceptable; (2) **Drop** — reject new messages at capacity; fast, no memory growth; acceptable for telemetry/metrics where losing some data is fine; never for payments; (3) **Sample** — accept 1 in N items when overloaded; preserves statistical shape; used in high-volume logging; (4) **Scale out** — autoscale consumers (Kubernetes HPA on queue depth); correct for payments but has lag; (5) **Circuit breaker** — stop accepting new work entirely; surface 503 to callers so they can retry later or fail gracefully. For payments: block + autoscale + alert; never drop; the SLA conversation determines which combination is acceptable.

---

### Q13. A checkout flow touches inventory, payment, and fulfillment services. How do you handle rollback when payment succeeds but order creation fails?

**Deep dive.** You can't use a distributed 2PC transaction — it holds locks across all services simultaneously, and a coordinator crash leaves everyone blocked forever. Instead: **Saga pattern**. A saga is a sequence of local transactions, each publishing an event or returning a result to the next step. On failure, previously completed steps are compensated (undone) by explicit compensating transactions in reverse order. Example: Reserve Inventory → Charge Payment → Create Order [FAILS] → Compensate: Refund Payment → Release Inventory. Compensation is not a database ROLLBACK — it's new business logic (a Stripe refund API call, a new inventory UPDATE). Two styles: **orchestration** (a central saga orchestrator sends commands and handles failures — clear flow, easy to trace, but orchestrator is a potential SPOF); **choreography** (services react to events — loose coupling, but failure flows are hard to trace across services). Gotchas: sagas have no isolation guarantee — a concurrent saga might read inventory that the first saga will compensate, leading to inconsistency. Accept this and design idempotent compensations with retry logic.

---

### Q14. You're designing an API gateway. What are its responsibilities, and why does it exist as a separate layer?

**Deep dive.** An API gateway is the single entry point for all external traffic. Its responsibilities: (1) **routing** — map `/api/orders/*` to the order service, `/api/products/*` to the catalog service; (2) **auth** — validate JWT/API key once at the edge, so every microservice doesn't re-implement auth; (3) **rate limiting** — enforce per-user/IP/API-key quotas before requests reach services; (4) **SSL termination** — decrypt HTTPS once at the gateway, internal traffic can be plain HTTP; (5) **request transformation** — inject headers, strip sensitive fields, aggregate responses; (6) **observability** — one place to log all request latency, error rates, add trace IDs. Without it: every microservice must implement auth, rate limiting, TLS separately — 10 services = 10 implementations, all potentially inconsistent. The gateway also enables gradual deploys (route 5% of traffic to v2) and A/B testing. Production: AWS API Gateway, Kong, Nginx, Envoy, or a custom FastAPI gateway service. Service discovery integrates here — the gateway queries the service registry to find healthy instances of each service.

---


---
## Scenario-Based Code Questions -- System Design

### Scenario 1 -- Token Bucket Rate Limiter

**Context:** ShopFlow's API gateway rate-limits to `max_tokens` requests/sec using token bucket (allows short bursts).

**Critical:** Use `time.monotonic()` NOT `time.time()` -- immune to NTP jumps!

In [ ]:
# -- SOLUTION --
import time, threading
from collections import defaultdict

class TokenBucket:
    def __init__(self, max_tokens: float, refill_rate: float):
        self.max, self.rate = max_tokens, refill_rate
        self._tokens: dict = defaultdict(lambda: max_tokens)
        self._last:   dict = defaultdict(lambda: 0.0)   # start at epoch 0 for predictable tests
        self._lock   = threading.Lock()

    def consume(self, user_id: str, n: float = 1, now: float | None = None) -> bool:
        now = now if now is not None else time.monotonic()
        with self._lock:
            elapsed = max(0.0, now - self._last[user_id])
            self._tokens[user_id] = min(self.max, self._tokens[user_id] + elapsed * self.rate)
            self._last[user_id]   = now
            if self._tokens[user_id] >= n:
                self._tokens[user_id] -= n; return True
            return False

tb = TokenBucket(max_tokens=10, refill_rate=5)
assert tb.consume('u1', 5, now=0.0)           # 10 → 5 remaining
assert not tb.consume('u1', 6, now=0.0)        # only 5 left, need 6 → denied
assert tb.consume('u1', 10, now=1.0)           # +5 refilled → 10 tokens; consume 10
print('Token bucket assertions passed ✓')


### Scenario 2 -- Bloom Filter

**Context:** BuildFast deduplicates build events before querying the DB. False positives OK (will re-check DB), false negatives NOT OK.

In [ ]:
# -- SOLUTION --
import math, hashlib

class BloomFilter:
    def __init__(self, capacity: int, error_rate: float = 0.01):
        self.size   = int(-capacity * math.log(error_rate) / (math.log(2)**2))
        self.hashes = int(self.size / capacity * math.log(2))
        self._bits  = bytearray(self.size)

    def _pos(self, item):
        return [int(hashlib.sha256(f'{i}:{item}'.encode()).hexdigest(), 16) % self.size
                for i in range(self.hashes)]

    def add(self, item: str):
        for p in self._pos(item): self._bits[p] = 1

    def might_contain(self, item: str) -> bool:
        return all(self._bits[p] for p in self._pos(item))

bf = BloomFilter(capacity=10_000, error_rate=0.01)
for i in range(1000): bf.add(f'event_{i}')
assert bf.might_contain('event_0') and bf.might_contain('event_999')
assert not bf.might_contain('event_99999')
fps = sum(1 for i in range(10000, 20000) if bf.might_contain(f'event_{i}'))
print(f'False positive rate: {fps/10000:.2%} (target: ~1%)')

---
## Part 5 — Knowledge Check (Advanced Topics)

**1. N=4 nodes with quorum=3 survives how many failures?**
- A) 2
- B) 3
- C) 1
- D) 0

**2. The zombie leader problem is best prevented by:**
- A) Longer heartbeat intervals
- B) Fencing tokens (monotonic election IDs) rejected by storage if stale
- C) Running 2PC instead of Raft
- D) Restarting the old leader

**3. A Saga's compensating transaction is:**
- A) A database ROLLBACK across services
- B) New business logic that explicitly undoes the previous step (e.g., a refund API call)
- C) A 2-phase commit prepare phase
- D) An idempotency key lookup

**4. Connection pool size should be approximately:**
- A) max_connections of the DB
- B) Number of concurrent users
- C) num_CPU_cores × 2 + num_disks (per PostgreSQL guidance)
- D) Unlimited — let the pool grow dynamically

**5. During a partition, Cassandra (AP) continues accepting writes. The tradeoff is:**
- A) Writes are silently dropped
- B) Reads may return stale data until nodes reconcile
- C) The cluster elects a new primary
- D) Writes require quorum

**6. An idempotency key in Redis must use which atomic operation to prevent concurrent duplicate execution?**
- A) GET then SET
- B) SET NX (set-if-not-exists)
- C) INCR
- D) LPUSH

**7. The Dead Letter Queue is used when:**
- A) The queue is empty
- B) A message fails after N retries and must be parked for inspection rather than dropped
- C) Consumers are faster than producers
- D) Messages expire their TTL without being consumed once

**8. Backpressure's "drop" strategy is appropriate for:**
- A) Payment processing
- B) Order creation
- C) High-volume telemetry/metrics where some data loss is acceptable
- D) Database writes

**9. An API gateway's primary auth responsibility is:**
- A) Encrypting DB queries
- B) Validating JWT/API keys once at the edge so individual services don't re-implement auth
- C) Storing user passwords
- D) Managing DB connection pools

**10. Service discovery with TTL-based health checks means:**
- A) Services are permanently registered and never expire
- B) A service entry expires unless renewed by heartbeat — dead services auto-deregister
- C) The gateway polls services every second
- D) Services register only when health checks pass

### Answer Key
1. **C** — N=4, quorum=3, tolerates 1 failure (identical to N=3; N=4 wastes a server).
2. **B** — Fencing tokens; storage rejects stale-token writes from the zombie leader.
3. **B** — Compensation is new business logic, not a DB rollback.
4. **C** — PostgreSQL's own formula; divide total by app instances for per-pool size.
5. **B** — AP systems stay available but may serve stale reads until anti-entropy reconciles.
6. **B** — `SET NX` is atomic; GET+SET has a race condition where two retries both execute.
7. **B** — DLQ parks poison-pill or persistently-failing messages for human review + replay.
8. **C** — Drop is only safe when losing individual data points doesn't affect correctness.
9. **B** — Auth at the gateway = one implementation, consistent enforcement across all services.
10. **B** — TTL + heartbeat = passive health check; expired entries are auto-removed from the registry.


---
## Scenario-Based Code Questions — Advanced System Design

*Each scenario puts you in a real on-call or design situation. Write the code first, then check the solution.*


### Scenario 3 — Thread-Safe Connection Pool

**Context:** ShopFlow's order service opens a new DB connection per request. Under 500 concurrent users the DB crashes (too many connections). Implement a thread-safe `ConnectionPool` with `acquire()` as a context manager. Pool must block if all connections are in use, and raise `TimeoutError` if no connection is available within `timeout` seconds. Include connection health validation before hand-off.

**Constraints:**
- Fixed pool size (no dynamic growth)
- `acquire()` must be a context manager (`with pool.acquire() as conn:`)
- Connections that error during use must be replaced with a fresh one, not returned as poisoned
- Must be thread-safe


In [ ]:
# -- SOLUTION --
import threading
import queue
import time
from contextlib import contextmanager

class FakeConnection:
    """Simulates a DB connection."""
    _counter = 0

    def __init__(self):
        FakeConnection._counter += 1
        self.id = FakeConnection._counter
        self.broken = False

    def is_alive(self) -> bool:
        return not self.broken

    def execute(self, sql: str) -> str:
        if self.broken:
            raise RuntimeError("Connection is broken")
        return f"conn#{self.id}: OK ({sql})"


class ConnectionPool:
    def __init__(self, factory, size: int = 10, timeout: float = 5.0):
        self._factory = factory
        self._timeout = timeout
        self._pool: queue.Queue = queue.Queue(maxsize=size)
        # Pre-warm the pool
        for _ in range(size):
            self._pool.put(factory())

    @contextmanager
    def acquire(self):
        try:
            conn = self._pool.get(timeout=self._timeout)
        except queue.Empty:
            raise TimeoutError(f"No connection available within {self._timeout}s")

        ok = True
        try:
            if not conn.is_alive():       # health check before hand-off
                conn = self._factory()    # replace broken connection
            yield conn
        except Exception:
            ok = False
            conn.broken = True            # mark broken; don't return to pool
            raise
        finally:
            if ok:
                self._pool.put(conn)      # return healthy connection
            else:
                self._pool.put(self._factory())  # replace with fresh one


pool = ConnectionPool(FakeConnection, size=3, timeout=1.0)

# Happy path
with pool.acquire() as c:
    print(pool.acquire.__doc__ or "ctx acquired")
    print(c.execute("SELECT 1"))

# Concurrent test: 3 threads use the pool simultaneously
results = []
def worker(n):
    with pool.acquire() as c:
        time.sleep(0.05)
        results.append(c.execute(f"SELECT {n}"))

threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()
print("Concurrent results:", results)

# Broken connection replacement
with pool.acquire() as c:
    c.broken = True
    try:
        c.execute("SELECT boom")
    except RuntimeError:
        pass  # triggers replacement

# Pool should have 3 healthy connections still
remaining = pool._pool.qsize()
print(f"Pool healthy after broken conn: {remaining == 3} (size={remaining})")
assert remaining == 3, f"Expected 3, got {remaining}"
print("All connection pool assertions passed ✓")


### Scenario 4 — Saga Orchestrator with Compensation

**Context:** ShopFlow checkout flow: (1) reserve inventory → (2) charge payment → (3) create order → (4) trigger fulfillment. If step N fails, steps 1…N-1 must be compensated in reverse. Implement a `SagaOrchestrator` that executes steps and auto-runs compensations on failure.

**Constraints:**
- Steps are `(name, action, compensation)` tuples where each is a callable
- On failure of step N, run compensations for steps N-1 down to 0
- Compensations must run even if a previous compensation raises
- Return a structured result showing which step failed and which compensations ran


In [ ]:
# -- SOLUTION --
from dataclasses import dataclass, field
from typing import Callable, Any

@dataclass
class SagaStep:
    name: str
    action: Callable[[], Any]
    compensation: Callable[[], None]

@dataclass
class SagaResult:
    success: bool
    completed_steps: list[str] = field(default_factory=list)
    failed_step: str | None = None
    compensated_steps: list[str] = field(default_factory=list)
    compensation_errors: list[str] = field(default_factory=list)
    error: Exception | None = None


class SagaOrchestrator:
    def execute(self, steps: list[SagaStep]) -> SagaResult:
        result = SagaResult(success=False)
        completed: list[SagaStep] = []

        for step in steps:
            try:
                step.action()
                completed.append(step)
                result.completed_steps.append(step.name)
            except Exception as exc:
                result.failed_step = step.name
                result.error = exc
                # Compensate in reverse order; never let one failure abort others
                for done in reversed(completed):
                    try:
                        done.compensation()
                        result.compensated_steps.append(done.name)
                    except Exception as comp_exc:
                        result.compensation_errors.append(
                            f"{done.name}: {comp_exc}"
                        )
                return result

        result.success = True
        return result


# ── Simulation ────────────────────────────────────────────────────────────────
log: list[str] = []

def make_checkout_steps(fail_at: str | None = None):
    def action(name):
        def _():
            if name == fail_at:
                raise RuntimeError(f"{name} failed")
            log.append(f"  ✓ {name}")
        return _

    def compensate(name):
        def _(): log.append(f"  ↩ compensate {name}")
        return _

    return [
        SagaStep("reserve_inventory", action("reserve_inventory"), compensate("reserve_inventory")),
        SagaStep("charge_payment",    action("charge_payment"),    compensate("charge_payment")),
        SagaStep("create_order",      action("create_order"),      compensate("create_order")),
        SagaStep("trigger_fulfillment",action("trigger_fulfillment"),compensate("trigger_fulfillment")),
    ]

orchestrator = SagaOrchestrator()

print("── Happy path ──────────────────────────────────")
log.clear()
r = orchestrator.execute(make_checkout_steps())
for l in log: print(l)
assert r.success and r.failed_step is None

print("\n── Payment fails → compensate inventory ────────")
log.clear()
r = orchestrator.execute(make_checkout_steps(fail_at="charge_payment"))
for l in log: print(l)
assert not r.success
assert r.failed_step == "charge_payment"
assert "reserve_inventory" in r.compensated_steps
assert "charge_payment" not in r.compensated_steps  # never ran

print("\n── Order creation fails → compensate payment + inventory ──")
log.clear()
r = orchestrator.execute(make_checkout_steps(fail_at="create_order"))
for l in log: print(l)
assert r.compensated_steps == ["charge_payment", "reserve_inventory"]

print("\nAll saga assertions passed ✓")


### Scenario 5 — Dead Letter Queue with Retry Logic

**Context:** BuildFast's build-event consumer processes messages from a queue. Transient failures should retry up to `max_retries` times with exponential backoff. After exhausting retries the message moves to a Dead Letter Queue (DLQ) instead of being dropped. Implement `MessageQueue`, `DeadLetterQueue`, and `Consumer` with retry + DLQ routing.

**Constraints:**
- Simulate with in-memory queues (no real broker needed)
- Exponential backoff: `base_delay * 2^attempt` (no actual sleep — use a `sleep_fn` hook for testing)
- DLQ must record: original message, failure reason, attempt count, and timestamp
- Consumer must be able to replay DLQ messages back to the main queue


In [ ]:
# -- SOLUTION --
import time
from collections import deque
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Message:
    id: str
    payload: dict
    attempt: int = 0

@dataclass
class DLQEntry:
    message: Message
    reason: str
    attempts: int
    timestamp: float = field(default_factory=time.monotonic)


class MessageQueue:
    def __init__(self): self._q: deque[Message] = deque()
    def publish(self, msg: Message): self._q.append(msg)
    def poll(self) -> Message | None:
        return self._q.popleft() if self._q else None
    def __len__(self): return len(self._q)


class DeadLetterQueue:
    def __init__(self): self._entries: list[DLQEntry] = []
    def park(self, msg: Message, reason: str):
        self._entries.append(DLQEntry(msg, reason, msg.attempt))
    def replay_all(self, target: MessageQueue):
        while self._entries:
            entry = self._entries.pop(0)
            entry.message.attempt = 0        # reset retry counter
            target.publish(entry.message)
    def __len__(self): return len(self._entries)
    def entries(self): return list(self._entries)


class Consumer:
    def __init__(
        self,
        queue: MessageQueue,
        dlq: DeadLetterQueue,
        handler: Callable[[Message], None],
        max_retries: int = 3,
        base_delay: float = 1.0,
        sleep_fn: Callable[[float], None] = time.sleep,
    ):
        self._queue = queue
        self._dlq = dlq
        self._handler = handler
        self._max_retries = max_retries
        self._base_delay = base_delay
        self._sleep = sleep_fn
        self.processed: list[str] = []

    def process_one(self) -> bool:
        msg = self._queue.poll()
        if msg is None:
            return False
        while msg.attempt <= self._max_retries:
            try:
                self._handler(msg)
                self.processed.append(msg.id)
                return True
            except Exception as exc:
                msg.attempt += 1
                if msg.attempt > self._max_retries:
                    self._dlq.park(msg, str(exc))
                    return True
                delay = self._base_delay * (2 ** (msg.attempt - 1))
                self._sleep(delay)
        return True


# ── Test ──────────────────────────────────────────────────────────────────────
sleeps: list[float] = []

main_q = MessageQueue()
dlq    = DeadLetterQueue()

# Message that always fails → goes to DLQ
fail_msg = Message(id="evt-001", payload={"type": "BUILD_FAILED"})
main_q.publish(fail_msg)

consumer = Consumer(
    main_q, dlq,
    handler=lambda m: (_ for _ in ()).throw(RuntimeError("DB down")),
    max_retries=2,
    base_delay=0.001,
    sleep_fn=lambda d: sleeps.append(d),
)
consumer.process_one()

assert len(dlq) == 1, "Should have 1 DLQ entry"
entry = dlq.entries()[0]
assert entry.message.id == "evt-001"
assert entry.attempts == 3  # initial + 2 retries
print(f"DLQ entry: id={entry.message.id}, attempts={entry.attempts}, reason='{entry.reason}'")
print(f"Backoff delays (s): {sleeps}")  # [0.001, 0.002]

# Message that succeeds → processed normally
ok_msg = Message(id="evt-002", payload={"type": "BUILD_OK"})
main_q.publish(ok_msg)
consumer_ok = Consumer(main_q, dlq, handler=lambda m: None, sleep_fn=lambda d: None)
consumer_ok.process_one()
assert "evt-002" in consumer_ok.processed

# Replay DLQ → messages go back to main queue
dlq.replay_all(main_q)
assert len(dlq) == 0
assert len(main_q) == 1  # evt-001 replayed
replayed = main_q.poll()
assert replayed.id == "evt-001" and replayed.attempt == 0, "attempt must reset on replay"

print("All DLQ assertions passed ✓")


### Scenario 6 — Service Registry with TTL-Based Health (Service Discovery)

**Context:** ShopFlow runs 4 microservices each with multiple instances. The API gateway must route to a healthy instance of each service. Implement a `ServiceRegistry` where services register with a TTL, renew via heartbeat, and auto-expire when unhealthy. The gateway queries the registry to load-balance (round-robin) across live instances.

**Constraints:**
- Registration stores `(service_name, instance_id, host, port, ttl_seconds)`
- Heartbeat renews the TTL; missing heartbeat within TTL = instance considered dead and removed
- `get_instance(service_name)` returns a healthy instance using round-robin (or raises if none available)
- `cleanup()` removes all expired instances (call before routing)


In [ ]:
# -- SOLUTION --
import time
import threading
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Callable

@dataclass
class ServiceInstance:
    service: str
    instance_id: str
    host: str
    port: int
    ttl: float
    _registered_at: float = field(default_factory=time.monotonic, repr=False)
    _last_heartbeat: float = field(default_factory=time.monotonic, repr=False)

    def renew(self, clock=time.monotonic): self._last_heartbeat = clock()

    def is_alive(self, now: float | None = None) -> bool:
        now = now if now is not None else time.monotonic()  # 0.0 is falsy; use explicit None check
        return (now - self._last_heartbeat) < self.ttl


class ServiceRegistry:
    def __init__(self, clock: Callable[[], float] = time.monotonic):
        self._instances: dict[str, dict[str, ServiceInstance]] = defaultdict(dict)
        self._lock = threading.Lock()
        self._clock = clock
        self._rr_counters: dict[str, int] = defaultdict(int)

    def register(self, service: str, instance_id: str, host: str, port: int, ttl: float = 30.0):
        t = self._clock()
        inst = ServiceInstance(service, instance_id, host, port, ttl,
                               _registered_at=t, _last_heartbeat=t)
        with self._lock:
            self._instances[service][instance_id] = inst

    def heartbeat(self, service: str, instance_id: str):
        with self._lock:
            inst = self._instances[service].get(instance_id)
            if inst:
                inst._last_heartbeat = self._clock()

    def deregister(self, service: str, instance_id: str):
        with self._lock:
            self._instances[service].pop(instance_id, None)

    def cleanup(self):
        """Remove expired instances; call before routing."""
        now = self._clock()
        with self._lock:
            for svc in list(self._instances):
                dead = [iid for iid, inst in self._instances[svc].items()
                        if not inst.is_alive(now)]
                for iid in dead:
                    del self._instances[svc][iid]

    def get_instance(self, service: str) -> ServiceInstance:
        self.cleanup()
        with self._lock:
            alive = list(self._instances[service].values())
        if not alive:
            raise LookupError(f"No healthy instances for '{service}'")
        idx = self._rr_counters[service] % len(alive)
        self._rr_counters[service] += 1
        return alive[idx]

    def list_healthy(self, service: str) -> list[ServiceInstance]:
        self.cleanup()
        with self._lock:
            return list(self._instances[service].values())


# ── Test ──────────────────────────────────────────────────────────────────────
_fake_time2 = [0.0]
def fake_clock2(): return _fake_time2[0]
def advance2(seconds): _fake_time2[0] += seconds

reg2 = ServiceRegistry(clock=fake_clock2)

for i in range(3):
    reg2.register("order-service", f"order-{i}", f"10.0.0.{i}", 8000 + i, ttl=30.0)

assert len(reg2.list_healthy("order-service")) == 3, \
    f"Expected 3, got {len(reg2.list_healthy('order-service'))}"

# Round-robin distributes across all instances
seen = {reg2.get_instance("order-service").instance_id for _ in range(6)}
assert len(seen) == 3, f"Round-robin should hit all 3, got {seen}"
print("Round-robin distribution:", seen)

# Advance time past TTL; manually renew only order-1 and order-2
advance2(31)
reg2._instances["order-service"]["order-1"]._last_heartbeat = fake_clock2()
reg2._instances["order-service"]["order-2"]._last_heartbeat = fake_clock2()

alive = reg2.list_healthy("order-service")
assert len(alive) == 2, f"Expected 2 alive, got {len(alive)}"
assert all(inst.instance_id != "order-0" for inst in alive)
print("After TTL expiry, alive instances:", [inst.instance_id for inst in alive])

try:
    reg2.get_instance("unknown-service")
    assert False, "Should raise LookupError"
except LookupError as e:
    print(f"Correct error: {e}")

print("All service registry assertions passed ✓")


### Scenario 7 — Backpressure Queue with Drop / Block Strategies

**Context:** ShopFlow's analytics pipeline receives product-view events from the API gateway. At peak (flash sales), events arrive 10× faster than the processor can handle. Implement a `BackpressureQueue` that supports two strategies: `"block"` (caller blocks until space) and `"drop"` (reject immediately at capacity). Include metrics: `accepted`, `dropped`, `processed`.

**Constraints:**
- Bounded capacity (fixed max size)
- `put(item)` behaves according to strategy
- `get()` blocks until an item is available (producer/consumer decoupled)
- Thread-safe
- `metrics()` returns a snapshot dict


In [ ]:
# -- SOLUTION --
import queue
import threading
from typing import Any, Literal

Strategy = Literal["block", "drop"]

class BackpressureQueue:
    def __init__(self, maxsize: int, strategy: Strategy = "drop"):
        if strategy not in ("block", "drop"):
            raise ValueError(f"Unknown strategy: {strategy!r}")
        self._q = queue.Queue(maxsize=maxsize)
        self._strategy = strategy
        self._accepted = 0
        self._dropped  = 0
        self._processed = 0
        self._lock = threading.Lock()

    def put(self, item: Any, timeout: float | None = None) -> bool:
        """Returns True if accepted, False if dropped."""
        if self._strategy == "drop":
            try:
                self._q.put_nowait(item)
                with self._lock: self._accepted += 1
                return True
            except queue.Full:
                with self._lock: self._dropped += 1
                return False
        else:  # block
            self._q.put(item, timeout=timeout)   # blocks; caller handles TimeoutError
            with self._lock: self._accepted += 1
            return True

    def get(self, timeout: float | None = None) -> Any:
        item = self._q.get(timeout=timeout)
        with self._lock: self._processed += 1
        return item

    def metrics(self) -> dict:
        with self._lock:
            return {
                "accepted": self._accepted,
                "dropped":  self._dropped,
                "processed": self._processed,
                "queue_depth": self._q.qsize(),
            }


# ── DROP strategy test ────────────────────────────────────────────────────────
bq_drop = BackpressureQueue(maxsize=3, strategy="drop")

# Fill to capacity
for i in range(3): assert bq_drop.put(f"event-{i}") is True
# Next 5 are dropped
for i in range(5): assert bq_drop.put(f"overflow-{i}") is False

m = bq_drop.metrics()
assert m["accepted"] == 3 and m["dropped"] == 5, m
print(f"DROP metrics: {m}")

# Consume all 3
for _ in range(3): bq_drop.get(timeout=0.1)
assert bq_drop.metrics()["processed"] == 3

# ── BLOCK strategy + producer/consumer thread test ────────────────────────────
bq_block = BackpressureQueue(maxsize=5, strategy="block")
results = []

def producer():
    for i in range(10):
        bq_block.put(i)

def consumer():
    for _ in range(10):
        results.append(bq_block.get(timeout=2.0))

t_prod = threading.Thread(target=producer)
t_cons = threading.Thread(target=consumer)
t_prod.start(); t_cons.start()
t_prod.join(); t_cons.join()

assert sorted(results) == list(range(10)), f"Expected 0-9, got {sorted(results)}"
m2 = bq_block.metrics()
assert m2["accepted"] == 10 and m2["dropped"] == 0 and m2["processed"] == 10
print(f"BLOCK metrics: {m2}")
print("All backpressure assertions passed ✓")


### Scenario 8 — Idempotent Event Consumer (Exactly-Once Processing)

**Context:** ShopFlow's order service consumes `ORDER_PLACED` events from Kafka. The network re-delivers the same event after a consumer restart. Implement an `IdempotentConsumer` that processes each event exactly once using a bounded seen-set stored in an in-memory dict (simulate Redis `SET NX`). 

**Constraints:**
- `process(event_id, handler)` runs `handler()` only on the first call for that `event_id`
- Subsequent calls with the same `event_id` return the cached result without calling `handler` again
- The seen-set must be **bounded** (LRU eviction when full — simulate a bounded Redis key space)
- `handler` failures must NOT be cached (only cache successful results)
- Thread-safe (multiple consumer threads running concurrently)


In [ ]:
# -- SOLUTION --
import threading
from collections import OrderedDict
from typing import Any, Callable

_SENTINEL = object()  # marks "currently processing" to detect concurrent duplicates

class IdempotentConsumer:
    def __init__(self, max_size: int = 10_000):
        self._max_size = max_size
        self._seen: OrderedDict[str, Any] = OrderedDict()  # LRU: most-recent at end
        self._lock = threading.Lock()

    def _set_nx(self, event_id: str, value: Any) -> bool:
        """Atomic set-if-not-exists; returns True if inserted."""
        with self._lock:
            if event_id in self._seen:
                return False
            if len(self._seen) >= self._max_size:
                self._seen.popitem(last=False)   # evict oldest (LRU)
            self._seen[event_id] = value
            return True

    def _update(self, event_id: str, value: Any):
        with self._lock:
            self._seen[event_id] = value
            self._seen.move_to_end(event_id)     # mark as recently used

    def _get(self, event_id: str) -> Any:
        with self._lock:
            return self._seen.get(event_id, _SENTINEL)

    def process(self, event_id: str, handler: Callable[[], Any]) -> tuple[Any, bool]:
        """
        Returns (result, was_duplicate).
        Raises if handler raises (and does NOT cache the failure).
        """
        # Try to claim this event_id with a sentinel (in-progress marker)
        if not self._set_nx(event_id, _SENTINEL):
            # Already seen — wait briefly if in-progress, then return cached
            cached = self._get(event_id)
            while cached is _SENTINEL:           # spin until first execution completes
                import time; time.sleep(0.001)
                cached = self._get(event_id)
            return cached, True

        try:
            result = handler()
        except Exception:
            # Remove the sentinel so future retries can try again
            with self._lock:
                self._seen.pop(event_id, None)
            raise

        self._update(event_id, result)
        return result, False


# ── Test ──────────────────────────────────────────────────────────────────────
call_count = 0

def charge_card():
    global call_count
    call_count += 1
    return {"status": "charged", "amount": 99.00}

consumer = IdempotentConsumer(max_size=100)

# First call executes handler
result, dup = consumer.process("order-abc-001", charge_card)
assert not dup and result["status"] == "charged"
assert call_count == 1

# Duplicate call returns cached result WITHOUT re-executing
result2, dup2 = consumer.process("order-abc-001", charge_card)
assert dup2 and result2 == result
assert call_count == 1, f"Handler called {call_count} times — should be 1!"
print(f"Duplicate correctly cached: {result2}, call_count={call_count}")

# Failed handler is NOT cached — retry can execute again
def flaky():
    raise RuntimeError("Payment gateway down")

try:
    consumer.process("order-xyz-002", flaky)
except RuntimeError:
    pass

assert consumer._get("order-xyz-002") is _SENTINEL.__class__ or \
       "order-xyz-002" not in consumer._seen, "Failure must not be cached"
# Now a successful retry works
call_count2 = 0
def ok(): global call_count2; call_count2 += 1; return "ok"
r, d = consumer.process("order-xyz-002", ok)
assert not d and r == "ok"
print(f"Retry after failure succeeded: {r}")

# LRU eviction: fill to max, oldest is evicted
small = IdempotentConsumer(max_size=3)
for i in range(4):
    small.process(f"evt-{i}", lambda i=i: i)
assert "evt-0" not in small._seen, "Oldest should be evicted"
assert "evt-3" in small._seen
print(f"LRU eviction working, seen keys: {list(small._seen.keys())}")

# Concurrent duplicate suppression
concurrent_consumer = IdempotentConsumer(max_size=100)
counter = {"n": 0}
def slow_handler():
    import time; time.sleep(0.05)
    counter["n"] += 1
    return "done"

threads = [threading.Thread(target=concurrent_consumer.process, args=("shared-evt", slow_handler))
           for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
assert counter["n"] == 1, f"Handler ran {counter['n']} times concurrently — should be 1"
print(f"Concurrent duplicate suppression: handler ran {counter['n']} time ✓")
print("All idempotency assertions passed ✓")


---
## Part 6 — Advanced Gotchas Checklist

### Scalability
- **Stateless first** — you cannot horizontally scale a service that stores session data in process memory. Move it to Redis before anything else.
- **Read replicas before sharding** — read replicas are zero-code-change; sharding rewrites your data model. Exhaust cheaper options first.
- **N=2 is useless for HA** — quorum requires majority; N=2 means any failure = no quorum = full outage. Always use 3, 5, or 7.

### CAP / Consistency
- **Eventual consistency ≠ "will eventually be correct"** — without anti-entropy / reconciliation, diverged replicas can stay diverged forever. You must build the reconciliation.
- **PACELC applies even without partitions** — Cassandra trades Latency for Consistency on every write; Spanner accepts latency to achieve global strong consistency.
- **CP doesn't mean always available** — CP systems refuse writes during a partition. Design clients to retry with backoff; surface 503 to users, not 500.

### Connection Pooling
- **Never set pool_size to "unlimited"** — the DB has a `max_connections` limit; unlimited pools migrate the OOM crash from your app to the DB.
- **Validate before use, not just at creation** — a connection can silently go stale (DB restart, network idle timeout). Check `is_alive()` before handing to caller.
- **Async pools ≠ thread pools** — asyncpg/aiomysql pools work per event loop; sharing them across threads corrupts state.

### Leader Election / Consensus
- **Never roll your own consensus** — use etcd `LeaseGrant` + `KeepAlive`, or ZooKeeper ephemeral nodes. Home-grown consensus has subtle split-brain bugs that only appear under real network conditions.
- **Fencing tokens must be enforced by storage** — the leader can't fence itself. The DB/object-store must reject writes with a stale token.
- **Election timeout >> network RTT** — if the election timeout is shorter than your typical p99 network delay, you'll get constant spurious re-elections (leader flapping).

### Idempotency
- **Cache failures explicitly or not at all** — accidentally caching a transient failure (e.g. a 503 from the payment provider) as the permanent result means future correct requests get a 503 from your cache, not the provider.
- **SET NX, not GET+SET** — GET then SET is a race condition; two concurrent retries can both see "not found" and both execute. `SET NX` is atomic.
- **TTL must exceed your retry window** — if you retry for 24h but the idempotency key expires in 1h, a late retry re-executes the side effect.

### Backpressure
- **Unbounded queues hide backpressure problems** — they let producers run ahead until OOM. Bound the queue; make the backpressure explicit.
- **Blocking in async code blocks the event loop** — `queue.Queue.put()` blocks the thread. In asyncio use `asyncio.Queue` with `await q.put()`.
- **"Drop" must be monitored** — silently dropping events without a counter makes capacity problems invisible. Always emit `events_dropped_total` to your metrics system.

### Dead Letter Queue
- **DLQ is not a trash can** — every DLQ entry represents a lost transaction. Alert on DLQ depth > 0, and build a replay UI before your first incident.
- **Poison pills block partitions** — in Kafka, one bad message at offset N blocks all messages at N+1, N+2… forever. Always have DLQ + max retry config set.
- **Replay must be idempotent** — DLQ replay re-processes messages that partially succeeded. Your handler must be idempotent or replay causes double-execution.

### Saga Pattern
- **Compensations are not rollbacks** — a compensation is new business logic: a Stripe refund API call, an inventory UPDATE, an email. Test them as carefully as the forward path.
- **Sagas have no isolation** — a concurrent saga can read data that the first saga is about to compensate. This is a known limitation; handle it at the application layer (e.g., soft-delete inventory during reservation).
- **Choreography debugging is painful** — in a 6-step event-driven saga, tracing a failure requires correlating events across 6 services and 6 event types. Use a correlation ID and distributed tracing from day one.
